# Attack 2 — Synonym Substitution

Reproduction of the word-substitution attack from Gloaguen et al. (ICLR 2026), Figure 3 (left).
MarkLLM `SynonymSubstitution` (unchanged, as imported by the official robustness script):
a random subset of words is replaced by a random WordNet synonym, with no part-of-speech
or context check. Ratios: 0.1–0.5, as in `scripts/run_robustness_eval.py`.

Detection: local detector on retokenized text (`src/detection.py`), threshold p < 0.01.
Runs locally on Mac (CPU).

## 1. Setup

In [1]:
import random
import sys
from importlib.metadata import version
from pathlib import Path

import nltk
import numpy as np
from nltk.corpus import wordnet


def find_project_root(start: Path) -> Path:
    """Walk up the folders until the official ETH clone is found."""
    for folder in [start, *start.parents]:
        if (folder / "diffusion-lm-watermark").exists():
            return folder
    raise FileNotFoundError("Project root not found.")


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from detection import ALPHA, detect, load_detector, load_jsonl, save_jsonl

SEED = 42
nltk.download("wordnet", quiet=True)

wm = load_jsonl(ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl")
baseline = load_jsonl(ROOT / "results" / "attacks" / "baseline_original.jsonl")
OUT_DIR = ROOT / "results" / "attacks" / "synonym_substitution"
tokenizer, detector = load_detector(ROOT)

print("Watermarked records:", len(wm), "| baseline records:", len(baseline))
for pkg in ["markllm", "nltk"]:
    print(f"{pkg}: {version(pkg)}")
print("WordNet loaded, synsets for 'loan':", len(wordnet.synsets("loan")))

/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Watermarked records: 600 | baseline records: 600
markllm: 0.1.5
nltk: 3.10.4
WordNet loaded, synsets for 'loan': 3


## 2. Pilot: 5 texts

Same 5 texts as the word-deletion pilot, ratios 0.1 and 0.3. We also count how many
words are actually replaceable (have a WordNet synonym) to check whether the
requested ratio can be reached.

In [2]:
from markllm.evaluation.tools.text_editor import SynonymSubstitution


def count_targets(text: str, ratio: float) -> tuple:
    """Replicate MarkLLM's logic: number of replaceable words and of words actually replaced."""
    words = text.split()
    n_replaceable = sum(
        1 for w in words if any(len(s.lemmas()) > 1 for s in wordnet.synsets(w))
    )
    n_target = min(int(ratio * len(words)), n_replaceable)
    return len(words), n_replaceable, n_target


PILOT_IDX = [0, 1, 200, 201, 400]
PILOT_RATIOS = [0.1, 0.3]
editors = {r: SynonymSubstitution(ratio=r) for r in PILOT_RATIOS}

for idx in PILOT_IDX:
    original = wm[idx]["completion"]
    base = baseline[idx]
    print("=" * 100)
    print(f"Text {idx} | {base['dataset']} | original: p={base['p_value']:.2e}  "
          f"green={base['green_fraction']:.3f}  scored={base['n_scored']}")
    print("ORIGINAL:", original[:300].replace("\n", "\\n"), "...")

    for ratio in PILOT_RATIOS:
        random.seed(SEED + idx)
        attacked = editors[ratio].edit(original)
        res = detect(attacked, tokenizer, detector)
        n_words, n_repl, n_target = count_targets(original, ratio)
        print(f"\n  ratio {ratio:.1f} | words {n_words} | replaceable {n_repl} | "
              f"replaced {n_target} ({n_target / n_words:.1%}) | "
              f"p={res['p_value']:.2e}  green={res['green_fraction']:.3f}  "
              f"detected={res['detected']}")
        print("  ATTACKED:", attacked[:300], "...")

W1112 14:30:08.959000 46391 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Text 0 | 2-1_longform_qa | original: p=2.81e-15  green=0.480  scored=252
ORIGINAL: The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall apart because they don't work together anymore, or maybe they've gotten too far awa ...

  ratio 0.1 | words 250 | replaceable 141 | replaced 25 (10.0%) | p=9.09e-13  green=0.450  detected=True
  ATTACKED: The collapse of the Soviet Union is when unitary of the largest superpowers, like type A large super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall asunder because they don't put to work together anymore, operating theat ...

  ratio 0.3 | words 250 | replaceable 141 | replaced 75 (30.0%) | p=1.50e-06  green=0.376  detected=True
  ATTACKED: The crumple of the Soviet Union is when i of the largest superpowers, l

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 3. Full run: 600 texts x 5 ratios

Ratios 0.1–0.5 as in the paper. No 0.0 control: it would apply the same whitespace
normalization as `word_deletion_r0.0`, already measured. We record the number of
replaceable words and of words actually replaced (capped by WordNet coverage).

In [3]:
RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]
MARKLLM_VERSION = version("markllm")

for ratio in RATIOS:
    out_path = OUT_DIR / f"synonym_substitution_r{ratio:.1f}.jsonl"
    editor = SynonymSubstitution(ratio=ratio)
    records = []
    for idx, row in enumerate(wm):
        original = row["completion"]
        random.seed(SEED + idx)
        attacked = editor.edit(original)
        res = detect(attacked, tokenizer, detector)
        n_words, n_repl, n_target = count_targets(original, ratio)
        base = baseline[idx]
        records.append({
            "idx": idx,
            "dataset": base["dataset"],
            "attack": "synonym_substitution",
            "ratio": ratio,
            "seed": SEED + idx,
            "markllm_version": MARKLLM_VERSION,
            "prompt": row["prompt"],
            "original": original,
            "attacked": attacked,
            "n_words_original": n_words,
            "n_words_attacked": len(attacked.split()),
            "n_replaceable": n_repl,
            "n_replaced": n_target,
            "original_p_value": base["p_value"],
            "original_detected": base["detected"],
            "original_green_fraction": base["green_fraction"],
            **res,
        })
    save_jsonl(records, out_path)
    n_det = sum(r["detected"] for r in records if r["detected"] is not None)
    capped = sum(r["n_replaced"] < int(ratio * r["n_words_original"]) for r in records)
    print(f"ratio {ratio:.1f}: detected {n_det}/600 | texts capped by WordNet: {capped}")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/synonym_substitution/synonym_substitution_r0.1.jsonl
ratio 0.1: detected 559/600 | texts capped by WordNet: 0


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/synonym_substitution/synonym_substitution_r0.2.jsonl
ratio 0.2: detected 536/600 | texts capped by WordNet: 0


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/synonym_substitution/synonym_substitution_r0.3.jsonl
ratio 0.3: detected 471/600 | texts capped by WordNet: 0


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/synonym_substitution/synonym_substitution_r0.4.jsonl
ratio 0.4: detected 392/600 | texts capped by WordNet: 6


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/synonym_substitution/synonym_substitution_r0.5.jsonl
ratio 0.5: detected 257/600 | texts capped by WordNet: 86


## 4. Summary

Same metrics as the word-deletion notebook, plus the share of words actually replaced
(it can be lower than the requested ratio when WordNet has too few synonyms).

In [4]:
from scipy.stats import binomtest

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return f"[{ci.low:.1%}, {ci.high:.1%}]"

print(f"{'ratio':>5} | {'detection rate':>24} | {'ASR (on 583)':>24} | "
      f"{'mean green':>10} | {'mean scored':>11} | {'replaced':>8} | {'capped':>6}")
for ratio in RATIOS:
    recs = load_jsonl(OUT_DIR / f"synonym_substitution_r{ratio:.1f}.jsonl")
    n_det = sum(r["detected"] for r in recs)
    init = [r for r in recs if r["original_detected"]]
    n_removed = sum(not r["detected"] for r in init)
    green = np.mean([r["green_fraction"] for r in recs])
    scored = np.mean([r["n_scored"] for r in recs])
    replaced = np.mean([r["n_replaced"] / r["n_words_original"] for r in recs])
    capped = sum(r["n_replaced"] < int(ratio * r["n_words_original"]) for r in recs)
    print(f"{ratio:5.1f} | {n_det/600:6.1%} {wilson(n_det, 600):>17} | "
          f"{n_removed/len(init):6.1%} {wilson(n_removed, len(init)):>17} | "
          f"{green:10.3f} | {scored:11.1f} | {replaced:8.1%} | {capped:6d}")

ratio |           detection rate |             ASR (on 583) | mean green | mean scored | replaced | capped
  0.1 |  93.2%    [90.9%, 94.9%] |   4.1%      [2.8%, 6.1%] |      0.410 |       199.3 |     9.7% |      0
  0.2 |  89.3%    [86.6%, 91.6%] |   8.2%     [6.3%, 10.7%] |      0.380 |       212.7 |    19.7% |      0
  0.3 |  78.5%    [75.0%, 81.6%] |  19.2%    [16.2%, 22.6%] |      0.355 |       223.4 |    29.7% |      0
  0.4 |  65.3%    [61.4%, 69.0%] |  32.8%    [29.1%, 36.7%] |      0.334 |       232.7 |    39.7% |      6
  0.5 |  42.8%    [38.9%, 46.8%] |  55.9%    [51.9%, 59.9%] |      0.315 |       240.5 |    49.2% |     86
